## Session 25 - Wheat-Seeds Dataset
#### Link of Dataset : https://www.kaggle.com/datasets/jmcaro/wheat-seedsuci

In [1]:
import pandas as pd
import joblib

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    RandomizedSearchCV
)

from sklearn.svm import SVC
from sklearn.ensemble import (
    RandomForestClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier
)

from sklearn.metrics import accuracy_score

from xgboost import XGBClassifier

In [2]:
# Load Dataset
df = pd.read_csv("seeds.csv")

# Display Dataset
print(df.head())

# Independent Features
X = df.drop("Type", axis=1)

# Dependent Feature
y = df["Type"] - 1

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Dataset Loaded Successfully")

    Area  Perimeter  Compactness  Kernel.Length  Kernel.Width  \
0  15.26      14.84       0.8710          5.763         3.312   
1  14.88      14.57       0.8811          5.554         3.333   
2  14.29      14.09       0.9050          5.291         3.337   
3  13.84      13.94       0.8955          5.324         3.379   
4  16.14      14.99       0.9034          5.658         3.562   

   Asymmetry.Coeff  Kernel.Groove  Type  
0            2.221          5.220     1  
1            1.018          4.956     1  
2            2.699          4.825     1  
3            2.259          4.805     1  
4            1.355          5.175     1  
Dataset Loaded Successfully


In [3]:
# ---------------- Manual Search ----------------

c_values = [1, 10, 20]
kernels = ["linear", "rbf"]

best_accuracy = 0
best_c = None
best_kernel = None

print("Manual Search Results\n")

for c in c_values:
    for kernel in kernels:
        svm = SVC(C=c, kernel=kernel)
        svm.fit(X_train, y_train)

        accuracy = svm.score(X_test, y_test)
        print(
            f"C={c}, Kernel={kernel} --> Accuracy={accuracy:.4f}"
        )

        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_c = c
            best_kernel = kernel

print("\nBest Parameters")
print("C :", best_c)
print("Kernel :", best_kernel)
print("Accuracy :", round(best_accuracy,4))

# ---------------- Grid Search ----------------

param_grid = {
    "C":[1,10,20],
    "kernel":["linear","rbf"]
}

svm_grid = GridSearchCV(
    SVC(),
    param_grid,
    cv=5
)

svm_grid.fit(X_train,y_train)

print("\nGrid Search Best Parameters")
print(svm_grid.best_params_)
print("Best Score :",round(svm_grid.best_score_,4))

# ---------------- Random Search ----------------

svm_random = RandomizedSearchCV(
    SVC(),
    param_grid,
    n_iter=5,
    cv=5,
    random_state=42
)

svm_random.fit(X_train,y_train)

print("\nRandom Search Best Parameters")
print(svm_random.best_params_)
print("Best Score :",round(svm_random.best_score_,4))

Manual Search Results

C=1, Kernel=linear --> Accuracy=0.8750
C=1, Kernel=rbf --> Accuracy=0.8250
C=10, Kernel=linear --> Accuracy=0.9000
C=10, Kernel=rbf --> Accuracy=0.8500
C=20, Kernel=linear --> Accuracy=0.9000
C=20, Kernel=rbf --> Accuracy=0.8750

Best Parameters
C : 10
Kernel : linear
Accuracy : 0.9

Grid Search Best Parameters
{'C': 10, 'kernel': 'linear'}
Best Score : 0.9623

Random Search Best Parameters
{'kernel': 'linear', 'C': 10}
Best Score : 0.9623


In [8]:
svm_model = SVC(
    C=svm_grid.best_params_["C"],
    kernel=svm_grid.best_params_["kernel"]
)

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

ada_model = AdaBoostClassifier(
    n_estimators=100,
    random_state=42
)

gb_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42
)

xgb_model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42,
    eval_metric="mlogloss"
)

models = {
    "SVM": svm_model,
    "Random Forest": rf_model,
    "AdaBoost": ada_model,
    "Gradient Boosting": gb_model,
    "XGBoost": xgb_model
}

comparison = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    comparison.append([name, accuracy])

comparison = pd.DataFrame(
    comparison,
    columns=["Model","Accuracy"]
)

print(comparison)

               Model  Accuracy
0                SVM     0.900
1      Random Forest     0.850
2           AdaBoost     0.925
3  Gradient Boosting     0.875
4            XGBoost     0.875


In [9]:
best_model_name = comparison.loc[
    comparison["Accuracy"].idxmax(),
    "Model"
]

print("Best Model :", best_model_name)
best_model = models[best_model_name]
joblib.dump(best_model,"best_model.pkl")

print("Best Model Saved Successfully.")

Best Model : AdaBoost
Best Model Saved Successfully.
